# gm/ID LUT Toolbox — LUT1, LUT2, LUT3 i LUT4

Ovaj notebook je UI i kratka dokumentacija za `gmid_toolbox.py`.
Potrebno je da se u istom direktorijumu nalaze `65nch.mat` i
`65pch.mat`.

PMOS naponi se uvek zadaju kao pozitivne magnitude
$V_{SG}$, $V_{SD}$ i $V_{BS}$. Dužina kanala se zadaje u nm.
Dozvoljeni nazivi tranzistora su `nch`/`nmos` i `pch`/`pmos`.

Četiri tražena LUT-a su:

\[
V_{GS}=\mathrm{LUT1}(tranz,g_m/I_D,V_{DS},V_{SB},L)
\]

\[
V_{DS}=\mathrm{LUT2}(tranz,g_m/I_D,V_{GS},V_{SB},L)
\]

\[
g_m/I_D=\mathrm{LUT3}(tranz,V_{GS},V_{DS},V_{SB},L)
\]

\[
g_{ds}/I_D=\mathrm{LUT4}(tranz,V_{GS},V_{DS},V_{SB},L).
\]


## 1. Učitavanje toolbox-a

Ako notebook nije pokrenut iz direktorijuma u kome su MAT fajlovi,
promeni `DATA_DIR` u sledećoj ćeliji.


In [ ]:
from pathlib import Path

from gmid_toolbox import GMIDToolbox, LUTError

DATA_DIR = Path.cwd()

tb = GMIDToolbox(
    nmos_mat=DATA_DIR / "65nch.mat",
    pmos_mat=DATA_DIR / "65pch.mat",
    nmos_variable="nch",
    pmos_variable="pch",
)

print(tb.info())


## 2. Direktni API

Redosled argumenata je isti kao u matematičkom zapisu. LUT2 može
imati više matematičkih rešenja jer $g_m/I_D(V_{DS})$ ne mora biti
monotona funkcija. Bez dodatne informacije vraća najmanji presek.
U sweep-u se ista grana može pratiti opcijom `preferred_vds=...`.


In [ ]:
# LUT1: VSG PMOS-a
vsg = tb.LUT1("pch", 10.0, 0.10, 0.0, 65)

# LUT2: VDS NMOS-a
vds = tb.LUT2("nch", 15.0, 0.50, 0.0, 100)

# LUT3 i LUT4: direktno iz terminalnih napona
gmid = tb.LUT3("nch", 0.50, 0.30, 0.0, 100)
gdsid = tb.LUT4("pch", 0.50, 0.30, 0.0, 100)

print(f"LUT1 -> VSG    = {vsg:.6f} V")
print(f"LUT2 -> VDS    = {vds:.6f} V")
print(f"LUT3 -> gm/ID  = {gmid:.6f} 1/V")
print(f"LUT4 -> gds/ID = {gdsid:.6f} 1/V")


## 3. Interaktivni UI

Za LUT1 su ulazi `gm/ID` i `VDS/VSD`; za LUT2 su `gm/ID` i
`VGS/VSG`; za LUT3 i LUT4 oba polja predstavljaju terminalne napone.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
except ImportError as exc:
    raise ImportError("Instaliraj ipywidgets da bi UI radio.") from exc


tranz_widget = widgets.Dropdown(
    options=[("NMOS (nch)", "nch"), ("PMOS (pch)", "pch")],
    value="nch",
    description="Tranzistor:",
    style={"description_width": "90px"},
)
lut_widget = widgets.Dropdown(
    options=[
        ("LUT1: VGS/VSG", "LUT1"),
        ("LUT2: VDS/VSD", "LUT2"),
        ("LUT3: gm/ID", "LUT3"),
        ("LUT4: gds/ID", "LUT4"),
    ],
    value="LUT1",
    description="Operacija:",
    style={"description_width": "90px"},
)
input_a = widgets.FloatText(value=15.0, description="gm/ID:")
input_b = widgets.FloatText(value=0.30, description="VDS [V]:")
body_widget = widgets.FloatText(value=0.0, description="VSB [V]:")
length_widget = widgets.FloatText(value=100.0, description="L [nm]:")
button = widgets.Button(description="Izračunaj", button_style="primary")
output = widgets.Output()


def refresh_labels(*_):
    pmos = tranz_widget.value == "pch"
    op = lut_widget.value
    body_widget.description = "VBS [V]:" if pmos else "VSB [V]:"

    if op == "LUT1":
        input_a.description = "gm/ID:"
        input_b.description = "VSD [V]:" if pmos else "VDS [V]:"
    elif op == "LUT2":
        input_a.description = "gm/ID:"
        input_b.description = "VSG [V]:" if pmos else "VGS [V]:"
    else:
        input_a.description = "VSG [V]:" if pmos else "VGS [V]:"
        input_b.description = "VSD [V]:" if pmos else "VDS [V]:"


def calculate(_):
    with output:
        clear_output()
        op = lut_widget.value
        tranz = tranz_widget.value
        a = input_a.value
        b = input_b.value
        body = body_widget.value
        length = length_widget.value

        try:
            if op == "LUT1":
                value = tb.LUT1(tranz, a, b, body, length)
                label = "VSG" if tranz == "pch" else "VGS"
                print(f"{label} = {value:.9f} V")
            elif op == "LUT2":
                value = tb.LUT2(tranz, a, b, body, length)
                label = "VSD" if tranz == "pch" else "VDS"
                print(f"{label} = {value:.9f} V")
            elif op == "LUT3":
                value = tb.LUT3(tranz, a, b, body, length)
                print(f"gm/ID = {value:.9f} 1/V")
            else:
                value = tb.LUT4(tranz, a, b, body, length)
                print(f"gds/ID = {value:.9f} 1/V")
        except Exception as exc:
            print(f"{type(exc).__name__}: {exc}")


tranz_widget.observe(refresh_labels, names="value")
lut_widget.observe(refresh_labels, names="value")
button.on_click(calculate)
refresh_labels()

display(widgets.VBox([
    widgets.HTML("<h3>gm/ID LUT1–LUT4 kalkulator</h3>"),
    tranz_widget,
    lut_widget,
    input_a,
    input_b,
    body_widget,
    length_widget,
    button,
    output,
]))


## 4. Kompatibilnost sa starim kodom

Stari pozivi `tb.vgs`, `tb.vds`, `tb.gds_id` i `tb.jd`, kao i aliasi
`tb.LUT_VGS`, `tb.LUT_VDS`, `tb.LUT_GDS_ID` i `tb.LUT_JD`, ostali su
dostupni. Novi LUT4 je namerno odvojen od starog `tb.gds_id`, jer LUT4
prima terminalne napone, dok stara funkcija prima zadati `gm/ID`.
